In [2]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model


In [ ]:

# Load both trained models
try:
    cnn_model = load_model('Models\mnist_models\cnn_model_03.keras')
    
    print("Model is loaded successfully")
except Exception as e:
    print(f"Error loading model: {e}")
    exit()



Model is loaded successfully


In [4]:

# Label mapping (A-Y skipping J)
label_map = {
    0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G', 7: 'H', 
    8: 'I', 9: 'K', 10: 'L', 11: 'M', 12: 'N', 13: 'O', 14: 'P', 
    15: 'Q', 16: 'R', 17: 'S', 18: 'T', 19: 'U', 20: 'V', 21: 'W', 
    22: 'X', 23: 'Y'
}

#label_map = {i: chr(65 + i) for i in range(25) if i != 9}


In [5]:

def preprocessing(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5,5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    kernel = np.ones((3,3), np.uint8)
    cleaned = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
    return cv2.resize(cleaned, (28,28)).reshape(1,28,28,1)/255.0

def predict(frame):
    processed = preprocessing(frame)
    pred = cnn_model.predict(processed, verbose=0)[0]
    confidence = np.max(pred)
    
    if confidence < 0.7:
        return {"letter": "?", "confidence": confidence}
    return {
        "letter": label_map[np.argmax(pred)],
        "confidence": confidence
    }

In [6]:

# Initialize webcam
cap = cv2.VideoCapture(0)
roi_size = 300
roi_x, roi_y = (640 - roi_size) // 2, 100  # Centered ROI

while True:
    ret, frame = cap.read()
    if not ret: break
    
    frame = cv2.flip(frame, 1)
    cv2.rectangle(frame, (roi_x, roi_y), (roi_x + roi_size, roi_y + roi_size), (255, 0, 0), 2)
    roi = frame[roi_y:roi_y + roi_size, roi_x:roi_x + roi_size]
    
    if roi.size > 0:
        result = predict(roi)
        cv2.putText(frame, f"{result['letter']} ({result['confidence']:.1%})", 
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, 
                   (0, 255, 0) if result['confidence'] > 0.7 else (0, 0, 255), 2)
        
        debug_thresh = cv2.adaptiveThreshold(
            cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY), 
            255, 
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 11, 2
        )
        debug_img = cv2.resize(debug_thresh, (300, 300), interpolation=cv2.INTER_NEAREST)
        cv2.imshow('Processed View', debug_img)
        
        model_view = preprocessing(roi).reshape(28,28)
        cv2.imshow('Model View', cv2.resize(model_view, (280,280)))
    
    cv2.imshow('Sign Language Recognition', frame)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()